## Нормализация атрибутов товарных карточек

### Что делает код

Код приводит `attributes` карточек к единому виду, чтобы одинаковые характеристики не отличались только способом записи.

Основные виды нормализации:

- единицы измерения;
- числовые значения физических величин;
- многомерные размеры;
- синонимичные названия атрибутов.

### Общий пример

`"ширина": "12 см"` → `"ширина, мм": "120"`

`"число ядер": "8"` → `"количество ядер": "8"`


### Физические величины и единицы измерения

Единица измерения определяется только в достаточно надёжных случаях.

`длина` + `1.5 м` → `длина, мм: 1500`

`ширина, см` + `25` → `ширина, мм: 250`

`мощность (кВт)` + `1.2` → `мощность, Вт: 1200`


### Стандартизация числовых величин

Разные единицы одной физической величины переводятся в общий формат.

`1.5 кг` → `1500 г`

`2.3 м` → `2300 мм`

`0.75 л` → `750 мл`


### Многомерные размеры

`"размер упаковки": "120×80×50 мм"`

↓

`"длина упаковки": "120 мм"`  
`"ширина упаковки": "80 мм"`  
`"высота упаковки": "50 мм"`


### Стандартизация названий атрибутов

`число ядер` → `количество ядер`

`страна изготовителя` → `страна производителя`

`масса упаковки` → `вес упаковки`

`тип сенсора` → `тип датчика`


### Итоговый пайплайн

1. Исходные `attributes`
2. Определение физических величин и единиц измерения
3. Стандартизация единиц и числовых значений
4. Разбор многомерных размеров
5. Нормализация синонимичных названий атрибутов
6. Получение нормализованных `attributes`

In [1]:
import pandas as pd
import json
import re
import sys
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf
from match import CONFIG_DIR, resolve_project_path

with initialize_config_dir(version_base=None, config_dir=str(CONFIG_DIR)):
    cfg = compose(config_name="e-cup-normalization")

cards = pd.read_parquet(resolve_project_path(cfg.path.cards))
labels = pd.read_parquet(resolve_project_path(cfg.path.labels))
synonyms_df = pd.read_parquet(resolve_project_path(cfg.path.synonyms))
unique_attributes = pd.read_parquet(resolve_project_path(cfg.path.stats))['attribute'].tolist()
output_path = resolve_project_path(cfg.path.output)

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 1000)

## Скачиваю недостающие библиотеки

In [2]:
!pip install pint
!pip install -q pymorphy3 ruwordnet
!ruwordnet download

Defaulting to user installation because normal site-packages is not writeable
  Using cached flexcache-0.3-py3-none-any.whl.metadata (7.0 kB)
  Using cached flexparser-0.4-py3-none-any.whl.metadata (18 kB)
Using cached flexcache-0.3-py3-none-any.whl (13 kB)
Using cached flexparser-0.4-py3-none-any.whl (27 kB)

   ------------- -------------------------- 1/3 [flexcache]
   -------------------------- ------------- 2/3 [pint]
   -------------------------- ------------- 2/3 [pint]
   -------------------------- ------------- 2/3 [pint]
   -------------------------- ------------- 2/3 [pint]
   -------------------------- ------------- 2/3 [pint]
   -------------------------- ------------- 2/3 [pint]
   -------------------------- ------------- 2/3 [pint]
   ---------------------------------------- 3/3 [pint]



  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\Samoylov_Nikita\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\Samoylov_Nikita\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
"ruwordnet" �� ���� ����७��� ��� ���譥�
��������, �ᯮ��塞�� �ணࠬ��� ��� ������ 䠩���.


## Стандартизирую и нормализую одномерные аттрибуты с физическими величинами и численными характеристиками

### Паттерн 1: keyword + подходящая единица измерения

Например:

`вес товара` + `2 кг` → `вес товара, г: 2000`

`длина кабеля` + `1.5 м` → `длина кабеля, мм: 1500`

`объем бака` + `3 л` → `объем бака, мл: 3000`

### Паттерн 2: единица измерения явно указана в названии атрибута

Например:

`ширина, см` + `25` → `ширина, мм: 250`

`мощность (кВт)` + `1.2` → `мощность, Вт: 1200`


In [3]:
keyword_possible_units = {
    # Геометрия
    "ширина": {"нм", "мкм", "мм", "см", "дм", "м", "км", "дюйм", "ft"},
    "высота": {"нм", "мкм", "мм", "см", "дм", "м", "дюйм", "ft", "u"},
    "длина": {"нм", "мкм", "мм", "см", "дм", "м", "км", "дюйм", "ft", "пог. м"},
    "глубина": {"мкм", "мм", "см", "дм", "м", "дюйм"},
    "толщина": {"нм", "мкм", "мм", "см", "м", "дюйм"},
    "диаметр": {"мкм", "мм", "см", "м", "дюйм", "ft"},
    "радиус": {"мм", "см", "м", "км", "дюйм"},
    "размер": {"мкм", "мм", "см", "м", "дюйм", "ft"},
    "габарит": {"мм", "см", "м", "дюйм"},
    "периметр": {"мм", "см", "м"},
    "окружность": {"мм", "см", "м"},
    "клиренс": {"мм", "см", "м"},
    "дорожный просвет": {"мм", "см", "м"},
    "колея": {"мм", "см", "м"},
    "база": {"мм", "см", "м"},
    "размах": {"мм", "см", "м"},
    "ход": {"мкм", "мм", "см", "м"},
    "шаг": {"мкм", "мм", "см", "м"},
    "зазор": {"мкм", "мм", "см", "м"},
    "угол": {"°", "град", "рад"},

    # Площадь
    "площадь": {
        "мм²", "см²", "дм²", "м²", "км²",
        "кв. мм", "кв. см", "кв. м",
        "га"
    },

    # Масса / механика
    "вес": {"мг", "г", "кг", "т", "lb", "oz", "карат"},
    "масса": {"мг", "г", "кг", "т", "lb", "oz"},
    "максимальный вес": {"г", "кг", "т", "lb"},
    "допустимый вес": {"г", "кг", "т", "lb"},

    "нагрузка": {"г", "кг", "т", "н", "кн", "кгс", "lb"},
    "грузоподъемность": {"кг", "т", "lb"},
    "грузоподъёмность": {"кг", "т", "lb"},

    "сила": {"н", "кн", "мн", "кгс"},
    "усилие": {"н", "кн", "мн", "кгс"},
    "крутящий момент": {"н·м", "нм", "кгс·м", "lb-ft"},

    # Объём
    "объем": {
        "мкл", "мл", "л",
        "мм³", "см³", "дм³", "м³",
        "куб. мм", "куб. см", "куб. м",
        "кб", "мб", "гб", "тб"
    },
    "объём": {
        "мкл", "мл", "л",
        "мм³", "см³", "дм³", "м³",
        "куб. мм", "куб. см", "куб. м",
        "кб", "мб", "гб", "тб"
    },

    "вместимость": {
        "мл", "л", "см³", "дм³", "м³",
        "шт", "бутылок", "листов", "комплектов", "чел"
    },

    "емкость": {
        "мкл", "мл", "л",
        "см³", "дм³", "м³",
        "мкф", "нф", "пф", "ф",
        "мач", "ач",
        "вт·ч", "квт·ч",
        "мб", "гб", "тб",
        "шт"
    },
    "ёмкость": {
        "мкл", "мл", "л",
        "см³", "дм³", "м³",
        "мкф", "нф", "пф", "ф",
        "мач", "ач",
        "вт·ч", "квт·ч",
        "мб", "гб", "тб",
        "шт"
    },

    "резервуар": {"мл", "л", "см³", "дм³", "м³"},
    "бак": {"мл", "л", "см³", "дм³", "м³"},
    "объем бака": {"мл", "л", "см³", "дм³", "м³"},
    "объём бака": {"мл", "л", "см³", "дм³", "м³"},

    # Время
    "время": {"мкс", "мс", "с", "сек", "мин", "ч", "сут", "дни", "месяцы"},
    "длительность": {"мс", "с", "сек", "мин", "ч", "сут"},
    "продолжительность": {"мс", "с", "сек", "мин", "ч", "сут", "месяцы"},

    # Скорость
    "скорость": {
        "мм/с", "см/с", "м/с", "м/мин", "км/ч",
        "об/мин",
        "кадр/с", "стр/мин", "стежки/мин",
        "кб/с", "мб/с", "гб/с",
        "кбит/с", "мбит/с", "гбит/с",
        "iops", "ips"
    },

    "скорость вращения": {"об/мин", "об/с", "rpm"},
    "обороты": {"об/мин", "об/с", "rpm"},
    "число оборотов": {"об/мин", "об/с", "rpm"},

    "скорость передачи": {
        "бит/с", "кбит/с", "мбит/с", "гбит/с",
        "байт/с", "кб/с", "мб/с", "гб/с"
    },
    "скорость чтения": {"кб/с", "мб/с", "гб/с", "iops"},
    "скорость записи": {"кб/с", "мб/с", "гб/с", "iops"},
    "пропускная способность": {
        "бит/с", "кбит/с", "мбит/с", "гбит/с",
        "байт/с", "кб/с", "мб/с", "гб/с"
    },
    "битрейт": {"бит/с", "кбит/с", "мбит/с", "гбит/с"},

    # Расход / производительность
    "расход": {
        "мл/мин", "мл/ч",
        "л/мин", "л/ч", "л/сут",
        "м³/мин", "м³/ч", "м³/сут",
        "г/мин", "г/ч",
        "кг/мин", "кг/ч", "кг/сут",
        "л/кг", "л/м²", "кг/м²"
    },

    "производительность": {
        "г/мин", "г/ч",
        "кг/мин", "кг/ч", "кг/сут",
        "мл/мин", "л/мин", "л/ч",
        "м³/мин", "м³/ч", "м³/сут",
        "шт/мин", "шт/ч"
    },

    "расход воздуха": {"л/мин", "л/с", "м³/мин", "м³/ч", "cfm"},
    "расход воды": {"мл/мин", "л/мин", "л/ч", "м³/ч"},
    "расход топлива": {"л/ч", "л/100км", "кг/ч"},
    "воздушный поток": {"л/с", "л/мин", "м³/мин", "м³/ч", "cfm"},

    # Электрика
    "мощность": {"мвт", "вт", "квт", "мвт", "ва", "ква", "л.с.", "btu/h"},
    "потребляемая мощность": {"мвт", "вт", "квт", "ва", "ква"},
    "выходная мощность": {"мвт", "вт", "квт", "мвт", "ва", "ква", "л.с."},

    "напряжение": {"мв", "в", "кв"},
    "ток": {"мка", "ма", "а", "ка"},
    "сила тока": {"мка", "ма", "а", "ка"},
    "сопротивление": {"мом", "ом", "ком", "мом"},
    "заряд": {"мккл", "мкл", "кл", "мач", "ач"},

    "электрическая емкость": {"пф", "нф", "мкф", "мф", "ф"},
    "электрическая ёмкость": {"пф", "нф", "мкф", "мф", "ф"},

    # Энергия
    "энергия": {"дж", "кдж", "мдж", "вт·ч", "квт·ч", "кал", "ккал"},
    "энергопотребление": {"вт·ч", "квт·ч", "квт·ч/год"},

    # Частота
    "частота": {"гц", "кгц", "мгц", "ггц"},
    "частота вращения": {"об/мин", "об/с", "гц"},
    "частота обновления": {"гц"},
    "частота кадров": {"кадр/с", "fps"},

    # Температура
    "температура": {"°c", "°с", "°f", "k"},

    # Давление
    "давление": {
        "па", "кпа", "мпа",
        "бар", "мбар", "атм", "psi",
        "мм рт. ст.", "кгс/см²"
    },

    # Память / данные
    "память": {"бит", "байт", "кб", "мб", "гб", "тб"},
    "объем памяти": {"бит", "байт", "кб", "мб", "гб", "тб"},
    "объём памяти": {"бит", "байт", "кб", "мб", "гб", "тб"},
    "оперативная память": {"мб", "гб", "тб"},
    "накопитель": {"мб", "гб", "тб"},

    "разрядность": {"бит"},
    "битность": {"бит"},

    # Оптика / свет
    "световой поток": {"лм"},
    "освещенность": {"лк"},
    "освещённость": {"лк"},
    "яркость": {"кд/м²", "нит"},

    # Свойства вещества
    "плотность": {
        "мг/мл", "г/мл", "г/л",
        "г/см³", "кг/л", "кг/м³"
    },
    "вязкость": {"па·с", "мпа·с", "сст", "cst"},
    "твердость": {"shore a", "shore d", "hb", "hrc", "hv"},
    "твёрдость": {"shore a", "shore d", "hb", "hrc", "hv"},

    # Проценты / относительные характеристики
    "процент": {"%"},
    "доля": {"%"},
    "концентрация": {"%", "мг/мл", "мг/л", "г/л", "моль/л"},
    "влажность": {"%"},

    # Шум
    "уровень шума": {"дб", "дба"},
    "шум": {"дб", "дба"},
}





unit_aliases = {
    "мм": {
        "мм",
        "мм.",
        "миллиметр",
        "миллиметра",
        "миллиметров",
        "в мм",
        "в мм.",
        "в миллиметрах",
        "в милиметрах",
        "в миллииметрах",
    },

    "см": {
        "см",
        "см.",
        "сантиметр",
        "сантиметра",
        "сантиметров",
        "в см",
        "в сантиметрах",
        "в сантиметрх",
        "в сантиментрах",
        "в сантимерах",
    },

    "м": {
        "м",
        "метр",
        "метра",
        "метров",
        "в м",
        "в метрах",
    },

    "дюйм": {
        '"',
        "дюйм",
        "дюймы",
        "дюймов",
        "в дюймах",
        "(дюйм)",
    },

    "мл": {
        "мл",
        "мл.",
        "миллилитр",
        "миллилитров",
        "в мл",
        "в мл.",
        "в миллилитрах",
        "в милилитрах",
        "в миллиллитрах",
    },

    "л": {
        "л",
        "литр",
        "литров",
        "в л",
        "в литрах",
    },

    "г": {
        "г",
        "гр",
        "грамм",
        "граммов",
        "в г",
        "в гр",
        "в граммах",
    },

    "кг": {
        "кг",
        "килограмм",
        "килограммов",
        "в кг",
        "в килограммах",
        "в киллограммах",
        "в килогораммах",
    },

    "гб": {
        "гб",
        "гигабайт",
        "гигабайтов",
        "в гб",
        "в гигабайтах",
    },

    "мб": {
        "мб",
        "мегабайт",
        "мегабайтов",
        "в мб",
        "в мегабайтах",
    },

    "%": {
        "%",
        "процент",
        "процентов",
        "в %",
        "в процентах",
        "в массовых процентах",
    },
}



alias_to_unit = {
    alias.lower().strip(): canonical_unit
    for canonical_unit, aliases in unit_aliases.items()
    for alias in aliases
}



In [4]:
import re


# Исправил также пропущенную запятую после "артикулы"
skip_keywords = (
    "артикул",
    "артикулы",
    "oem",
    "sku",
    "part number",
    "partnumber",
    "партномер",
    "номер детали",
    "номер запчасти",
    "код товара",
    "код детали",
    "код производителя",
    "каталожный номер",
)


# ---------------------------------------------------------
# PRECOMPUTE
# ---------------------------------------------------------

# keyword -> regex самого keyword
_keyword_patterns = {
    keyword: re.compile(
        rf"(?<!\w){re.escape(keyword)}(?!\w)",
        flags=re.IGNORECASE,
    )
    for keyword in keyword_possible_units
}


# canonical unit -> aliases, длинные раньше
_unit_alias_lists = {
    canonical_unit: sorted(
        unit_aliases.get(canonical_unit, {canonical_unit}),
        key=len,
        reverse=True,
    )
    for units in keyword_possible_units.values()
    for canonical_unit in units
}


# keyword -> один regex, находящий любую допустимую единицу
_keyword_unit_patterns = {}

# keyword -> alias_lower -> canonical_unit
_keyword_alias_to_unit = {}

for keyword, allowed_units in keyword_possible_units.items():

    alias_map = {}

    for canonical_unit in allowed_units:
        for alias in _unit_alias_lists[canonical_unit]:
            alias_map[alias.lower().strip()] = canonical_unit

    aliases = sorted(
        alias_map,
        key=len,
        reverse=True,
    )

    if aliases:
        _keyword_unit_patterns[keyword] = re.compile(
            r"(?<!\w)("
            + "|".join(map(re.escape, aliases))
            + r")(?!\w)",
            flags=re.IGNORECASE,
        )

    _keyword_alias_to_unit[keyword] = alias_map


# canonical_unit -> regex для удаления этой ЕИ из key
_unit_cleanup_patterns = {}

for canonical_unit, aliases in _unit_alias_lists.items():
    alternatives = "|".join(
        map(
            re.escape,
            sorted(aliases, key=len, reverse=True),
        )
    )

    _unit_cleanup_patterns[canonical_unit] = (
        re.compile(
            rf"\(\s*(?:{alternatives})\s*\)",
            flags=re.IGNORECASE,
        ),
        re.compile(
            rf"(?<!\w)(?:{alternatives})(?!\w)",
            flags=re.IGNORECASE,
        ),
    )


# Один regex для явного ", unit"
_explicit_comma_pattern = re.compile(
    r",\s*("
    + "|".join(
        map(
            re.escape,
            sorted(alias_to_unit, key=len, reverse=True),
        )
    )
    + r")(?=$|\s)",
    flags=re.IGNORECASE,
)


# Один regex для "(unit)"
_explicit_bracket_pattern = re.compile(
    r"\(\s*("
    + "|".join(
        map(
            re.escape,
            sorted(alias_to_unit, key=len, reverse=True),
        )
    )
    + r")\s*\)",
    flags=re.IGNORECASE,
)


_space_pattern = re.compile(r"\s+")


# ---------------------------------------------------------
# FAST FUNCTION
# ---------------------------------------------------------

def normalize_physical_unit_pair(old_key, old_value):
    key = str(old_key).strip()
    value = str(old_value).strip()

    key_lower = key.lower()

    if any(word in key_lower for word in skip_keywords):
        return key, value


    # 1. keyword + подходящая единица
    for keyword, keyword_pattern in _keyword_patterns.items():

        if keyword_pattern.search(key) is None:
            continue

        unit_pattern = _keyword_unit_patterns.get(keyword)

        if unit_pattern is None:
            continue


        # Сначала ищем ЕИ в value
        match = unit_pattern.search(value)

        if match is not None:
            alias = match.group(1).lower().strip()

            canonical_unit = _keyword_alias_to_unit[keyword][alias]

            new_value = (value[:match.start()] + value[match.end():]).strip()

            clean_key = key

            bracket_pattern, standalone_pattern = _unit_cleanup_patterns[canonical_unit]

            clean_key = bracket_pattern.sub("", clean_key)
            clean_key = standalone_pattern.sub("", clean_key)

            clean_key = _space_pattern.sub(" ", clean_key).strip(" ,()")

            return f"{clean_key}, {canonical_unit}", new_value


        # Затем ищем ЕИ в key
        match = unit_pattern.search(key)

        if match is not None:
            alias = match.group(1).lower().strip()

            canonical_unit = keyword_alias_to_unit[keyword][alias]

            bracket_pattern, standalone_pattern = _unit_cleanup_patterns[canonical_unit]

            clean_key = bracket_pattern.sub("", key)

            clean_key = standalone_pattern.sub("", clean_key, count=1)

            clean_key = _space_pattern.sub(" ", clean_key).strip(" ,()")

            return f"{clean_key}, {canonical_unit}", value


    # 2. Явное ", ЕИ" / "(ЕИ)"
    comma_match = _explicit_comma_pattern.search(key)
    bracket_match = _explicit_bracket_pattern.search(key)

    if comma_match is not None:
        match = comma_match
    elif bracket_match is not None:
        match = bracket_match
    else:
        return key, value

    alias = match.group(1).lower().strip()
    canonical_unit = alias_to_unit[alias]

    clean_key = key[:match.start()].strip()

    value_unit_pattern = re.compile(
        rf"(?<!\w){re.escape(alias)}(?!\w)",
        flags=re.IGNORECASE,
    )

    new_value = value_unit_pattern.sub("", value, count=1).strip()

    return f"{clean_key}, {canonical_unit}", new_value

## Стандартизация величин

### Например:

`1.5 кг` → `1500 г`

`2.3 м` → `2300 мм`

`0.75 л` → `750 мл`

`2 ч` → `120 мин`


In [5]:
# Русская каноническая ЕИ -> имя для Pint
ru_to_pint = {
    # Длина
    "нм": "nanometer",
    "мкм": "micrometer",
    "мм": "millimeter",
    "см": "centimeter",
    "дм": "decimeter",
    "м": "meter",
    "км": "kilometer",
    "дюйм": "inch",
    "ft": "foot",

    # Масса
    "мг": "milligram",
    "г": "gram",
    "кг": "kilogram",
    "т": "metric_ton",
    "lb": "pound",
    "oz": "ounce",

    # Объём
    "мкл": "microliter",
    "мл": "milliliter",
    "л": "liter",
    "см³": "centimeter**3",
    "дм³": "decimeter**3",
    "м³": "meter**3",

    # Площадь
    "мм²": "millimeter**2",
    "см²": "centimeter**2",
    "м²": "meter**2",

    # Время
    "мс": "millisecond",
    "с": "second",
    "сек": "second",
    "мин": "minute",
    "ч": "hour",
    "сут": "day",

    # Мощность
    "мвт": "milliwatt",
    "вт": "watt",
    "квт": "kilowatt",

    # Напряжение
    "мв": "millivolt",
    "в": "volt",
    "кв": "kilovolt",

    # Ток
    "ма": "milliampere",
    "а": "ampere",

    # Частота
    "гц": "hertz",
    "кгц": "kilohertz",
    "мгц": "megahertz",
    "ггц": "gigahertz",

    # Давление
    "па": "pascal",
    "кпа": "kilopascal",
    "мпа": "megapascal",
    "бар": "bar",
    "атм": "atmosphere",
    "psi": "psi",

    # Энергия
    "дж": "joule",
    "кдж": "kilojoule",
    "вт·ч": "watt_hour",
    "квт·ч": "kilowatt_hour",

    # Сопротивление
    "ом": "ohm",
    "ком": "kiloohm",
    "мом": "megaohm",

    # Температура
    "°c": "degC",
    "°с": "degC",
    "°f": "degF",
    "к": "kelvin",
}


# Для каждого keyword: в какую ЕИ приводим.
# Слева/справа всё остаётся русским.
keyword_standard_unit = {
    "ширина": "мм",
    "высота": "мм",
    "длина": "мм",
    "глубина": "мм",
    "толщина": "мм",
    "диаметр": "мм",
    "радиус": "мм",
    "размер": "мм",
    "габарит": "мм",
    "периметр": "мм",
    "окружность": "мм",
    "клиренс": "мм",
    "дорожный просвет": "мм",
    "колея": "мм",
    "база": "мм",
    "размах": "мм",
    "ход": "мм",
    "шаг": "мм",
    "зазор": "мм",

    "площадь": "м²",

    "вес": "кг",
    "масса": "кг",
    "максимальный вес": "кг",
    "допустимый вес": "кг",

    "объем": "л",
    "объём": "л",
    "резервуар": "л",
    "бак": "л",
    "объем бака": "л",
    "объём бака": "л",

    "время": "с",
    "длительность": "с",
    "продолжительность": "с",

    "мощность": "вт",
    "потребляемая мощность": "вт",
    "выходная мощность": "вт",

    "напряжение": "в",
    "ток": "а",
    "сила тока": "а",

    "частота": "гц",
    "частота обновления": "гц",

    "давление": "па",
    "сопротивление": "ом",
    "энергия": "дж",

    "температура": "°c",
}

In [6]:
# Один раз до обработки карточек

_sorted_standard_keywords = sorted(
    keyword_standard_unit,
    key=len,
    reverse=True,
)

_standard_keyword_patterns = [
    (
        keyword,
        re.compile(
            rf"(?<!\w){re.escape(keyword)}(?!\w)",
            flags=re.IGNORECASE,
        ),
    )
    for keyword in _sorted_standard_keywords
]

In [7]:
from pint import UnitRegistry

_ureg = None

def get_ureg():
    global _ureg

    if _ureg is None:
        _ureg = UnitRegistry()

    return _ureg

# (from_unit, to_unit) -> (scale, offset)
_conversion_map = {}

ureg = get_ureg()

for from_unit, from_pint in ru_to_pint.items():
    for to_unit, to_pint in ru_to_pint.items():
        try:
            y0 = (0 * ureg(from_pint)).to(to_pint).magnitude
            y1 = (1 * ureg(from_pint)).to(to_pint).magnitude

            _conversion_map[(from_unit, to_unit)] = (y1 - y0, y0)
        except Exception:
            pass


def convert_physical_value(value, from_unit, to_unit):
    try:
        number = float(str(value).replace(",", "."))

        if from_unit == to_unit:
            return number

        conversion = _conversion_map.get((from_unit, to_unit))

        if conversion is None:
            return None

        scale, offset = conversion

        return number * scale + offset

    except Exception:
        return None


In [8]:
def standardize_physical_unit_pair(key, value):
    if "," not in key:
        return key, value

    attr, unit = map(str.strip, key.rsplit(",", 1))

    matched_keyword = None

    for keyword, pattern in _standard_keyword_patterns:
        if pattern.search(attr):
            matched_keyword = keyword
            break

    if matched_keyword is None:
        return key, value

    target_unit = keyword_standard_unit[matched_keyword]

    converted = convert_physical_value(value, unit.lower(), target_unit)

    if converted is None:
        return key, value

    if converted.is_integer():
        converted = int(converted)

    return f"{attr}, {target_unit}", str(converted)

In [9]:
def normalize_physical_attributes_dict(attrs):
    result = {}

    for old_key, old_value in attrs.items():
        key, value = normalize_physical_unit_pair(old_key, old_value)

        key, value = standardize_physical_unit_pair(key, value)

        result[key] = value

    return result

In [10]:
dimension_keywords = (
    "размер",
    "размеры",
    "габариты",
)

dimension_pattern = re.compile(
    r"^\s*"
    r"(\d+(?:[.,]\d+)?)"
    r"\s*[xх×*]\s*"
    r"(\d+(?:[.,]\d+)?)"
    r"(?:\s*[xх×*]\s*(\d+(?:[.,]\d+)?))?"
    r"\s*$",
    re.IGNORECASE,
)


def get_dimension_attribute_suffix(key):
    """
    'размер упаковки' -> 'упаковки'
    'размер товара'   -> 'товара'
    'размер'          -> None
    """
    match = re.search(
        r"(?<!\w)размер(?:ы)?\s+([а-яёa-z0-9_-]+)",
        str(key),
        flags=re.IGNORECASE,
    )

    return match.group(1).lower() if match else None


def normalize_multidimensional_attributes_dict(attrs):
    result = {}

    for old_key, old_value in attrs.items():
        old_key = str(old_key).strip()
        old_value = str(old_value).strip()

        if (len(old_value) > 20 or not any(contains_standalone_keyword(old_key, keyword) for keyword in dimension_keywords)):
            result[old_key] = old_value
            continue

        normalized_key, normalized_value = normalize_physical_unit_pair(old_key, old_value)

        if "," not in normalized_key:
            result[old_key] = old_value
            continue

        _, unit = map(str.strip, normalized_key.rsplit(",", 1))

        match = dimension_pattern.fullmatch(str(normalized_value))

        if match is None:
            result[old_key] = old_value
            continue

        a, b, c = match.groups()

        suffix = get_dimension_attribute_suffix(old_key)

        for name, value in zip(("длина", "ширина", "высота"), (a, b, c)):
            if value is None:
                continue

            if suffix:
                name = f"{name} {suffix}"

            key = f"{name}, {unit}"

            new_key, new_value = standardize_physical_unit_pair(key, value)

            result[new_key] = new_value

    return result

## Стандартизация названий аттрибутов

### Например:

`число ядер` → `количество ядер`

`страна изготовителя` → `страна производителя`

`масса упаковки` → `вес упаковки`

`тип сенсора` → `тип датчика`


In [11]:
import json
import re
import pandas as pd
import pymorphy3
from joblib import Parallel, delayed
import os


# synonym lemma -> replacer lemma
replace_map = {
    synonym: row.replacer
    for row in synonyms_df.itertuples(index=False)
    for synonym in row.synonyms
}


# Эти объекты будут отдельными в каждом worker-процессе
_morph = None
_synonym_word_cache = {}


def get_morph():
    global _morph

    if _morph is None:
        _morph = pymorphy3.MorphAnalyzer()

    return _morph


def normalize_synonym_word(word):
    word = word.lower()

    if word in _synonym_word_cache:
        return _synonym_word_cache[word]

    morph = get_morph()

    source = morph.parse(word)[0]
    lemma = source.normal_form

    replacer = replace_map.get(lemma)

    if replacer is None:
        result = word

    else:
        target = morph.parse(replacer)[0]

        grammemes = {
            x for x in (
                source.tag.case,
                source.tag.number,
                source.tag.gender,
            )
            if x is not None
        }

        inflected = target.inflect(grammemes)

        result = (
            inflected.word
            if inflected is not None
            else replacer
        )

    _synonym_word_cache[word] = result
    return result


def normalize_attribute_name_synonyms(text):
    return re.sub(
        r"[а-яё]+",
        lambda m: normalize_synonym_word(m.group()),
        str(text).lower(),
    )


def normalize_synonym_attributes_dict(attrs):
    normalized = {}

    for attr, value in attrs.items():
        new_attr = attribute_name_map.get(attr, attr)

        if new_attr not in normalized:
            normalized[new_attr] = value

    return normalized




attribute_name_map = {
    attr: normalize_attribute_name_synonyms(attr)
    for attr in unique_attributes
}

# Функция нормализации

In [12]:
from joblib import Parallel, delayed
import os
import time


def normalize_product_card_attributes(raw):
    try:
        attrs = json.loads(raw)
    except Exception:
        return raw

    try:
        attrs = normalize_multidimensional_attributes_dict(attrs)
    except Exception:
        pass

    try:
        attrs = normalize_physical_attributes_dict(attrs)
    except Exception:
        pass

    try:
        attrs = normalize_synonym_attributes_dict(attrs)
    except Exception:
        pass

    try:
        return json.dumps(
            attrs,
            ensure_ascii=False,
        )
    except Exception:
        return raw

In [13]:
import time
import numpy as np
from joblib import Parallel, delayed


def normalize_chunk(chunk):
    return [normalize_product_card_attributes(raw) for raw in chunk]


# Настройки
n_jobs = 2
chunk_size = 5_000

values = cards["attributes"].to_numpy()

chunks = [values[i:i + chunk_size] for i in range(0, len(values), chunk_size)]


start = time.perf_counter()

results = Parallel(n_jobs=n_jobs, backend="loky", pre_dispatch=n_jobs)(delayed(normalize_chunk)(chunk) for chunk in chunks)


normalized = [item for chunk in results for item in chunk]

cards["normalized_attributes"] = normalized

elapsed = time.perf_counter() - start

print(f"Карточек обработано: {len(cards):,}")
print(f"Время работы: {elapsed:.2f} сек")
print(f"Скорость: {len(cards) / elapsed:.0f} карточек/с")


cards.to_parquet(
    output_path,
    index=False,
)

print(f"Сохранено: {output_path}")

Карточек обработано: 711,304
Время работы: 127.56 сек
Скорость: 5576 карточек/с
Сохранено: C:\Users\Samoylov_Nikita\Documents\Twin2Attr\Twin2Attr\data\items_human_normalized.parquet


## Итого затронута обработками

In [14]:
import json

def count_changed_attributes(row):
    original = json.loads(row["attributes"])
    normalized = json.loads(row["normalized_attributes"])

    unchanged = sum(
        key in normalized and normalized[key] == value
        for key, value in original.items()
    )

    return len(original), len(original) - unchanged


stats = cards.apply(count_changed_attributes, axis=1, result_type="expand")

stats.columns = ["total", "changed"]

total_attributes = stats["total"].sum()
changed_attributes = stats["changed"].sum()
changed_percent = changed_attributes / total_attributes * 100

print(f"Всего атрибутов: {total_attributes:,}")
print(f"Затронуто нормализацией: {changed_attributes:,}")
print(f"Доля затронутых: {changed_percent:.2f}%")
print(f"Не затронуто: {total_attributes - changed_attributes:,}")
print(f"Доля не затронутых: {100 - changed_percent:.2f}%")

Всего атрибутов: 8,968,263
Затронуто нормализацией: 2,813,590
Доля затронутых: 31.37%
Не затронуто: 6,154,673
Доля не затронутых: 68.63%
